# Reproduce the reported results

This notebook verifies the deposited scientific inputs and regenerates the result files used for the reported portfolio-validity analysis. It runs directly from the public repository and requires no manual data upload.

Historical sample-selection verification is available separately in `selection-audit.ipynb`.


In [ ]:
from pathlib import Path
import os
import subprocess

repo_url = 'https://github.com/evidenceworks/university-portfolio-validity'
repo_name = 'university-portfolio-validity'
cwd = Path.cwd()

if (cwd / 'code' / 'check.py').is_file():
    repo = cwd
elif (cwd / repo_name / 'code' / 'check.py').is_file():
    repo = cwd / repo_name
else:
    base = Path('/content') if Path('/content').is_dir() else cwd
    repo = base / repo_name
    subprocess.run(['git', 'clone', '--depth', '1', repo_url, str(repo)], check=True)

os.chdir(repo)
print('Repository ready.')


## Run the checks and reproduction

The deposited inputs are checked before and after deterministic regeneration of the result files.


In [ ]:
import time

commands = (
    ['python', 'code/check.py'],
    ['python', 'code/reproduce.py'],
    ['python', 'code/check.py'],
)

started = time.perf_counter()
for command in commands:
    print('$', ' '.join(command))
    subprocess.run(command, check=True)
elapsed = time.perf_counter() - started

print()
print('Reproduction complete')
print('Input and cross-file checks: passed')
print('Result regeneration: passed')
print('Post-reproduction checks: passed')
print(f'Elapsed time: {elapsed:.1f} seconds')


## Main reproduced results

The table below is read from the regenerated `results/summary.csv` file.


In [ ]:
import pandas as pd

summary_table = pd.read_csv('results/summary.csv', dtype={'metric': str, 'value': str})
summary = summary_table.set_index('metric')['value'].to_dict()

def count_value(metric):
    return f"{int(float(summary[metric])):,}"

def percent_value(metric, digits=1):
    return f"{100 * float(summary[metric]):.{digits}f}%"

main_rows = [
    ('Audited cases', count_value('audited_cases')),
    ('Grade A', count_value('valid_a')),
    ('Grade B', count_value('valid_b')),
    ('Invalid', count_value('invalid')),
    ('Historically supported', count_value('accepted')),
    ('Adjudicated eligible', count_value('eligible')),
    ('Shared membership', count_value('shared')),
    ('Accepted only', count_value('accepted_only')),
    ('Eligible only', count_value('eligible_only')),
    ('Neither', count_value('neither')),
    (
        'Symmetric membership difference',
        f"{count_value('membership_difference')} records ({percent_value('membership_difference_rate')})",
    ),
    ('Jaccard overlap', percent_value('jaccard_overlap')),
    ('Accepted precision', percent_value('accepted_precision')),
    ('Eligible retention', percent_value('eligible_retention')),
    ('Exact four-label reviewer agreement', percent_value('reviewer_exact_agreement_rate')),
    (
        'Collapsed eligible/non-eligible reviewer agreement',
        percent_value('reviewer_collapsed_agreement_rate'),
    ),
    ('Source-labeled conference papers', count_value('conference_source_labeled')),
    ('Adjudicated eligible conference-paper cases', count_value('conference_eligible')),
]

main_results = pd.DataFrame(main_rows, columns=['Measure', 'Reproduced value'])
display(main_results.set_index('Measure'))


## Robustness and sensitivity

These diagnostics summarize institution-level influence, Grade B sensitivity and fixed publication-scale quintiles from the regenerated results.


In [ ]:
robustness = pd.read_csv('results/robustness.csv', dtype=str).fillna('')

def pct(raw):
    return f"{100 * float(raw):.1f}%"

def interval(analysis, metric):
    row = robustness[(robustness['analysis'] == analysis) & (robustness['group'] == metric)].iloc[0]
    return f"{pct(row['lower'])}–{pct(row['upper'])}"

cluster_row = robustness[
    (robustness['analysis'] == 'cluster_resampling')
    & (robustness['group'] == 'accepted_precision')
].iloc[0]

robust_rows = [
    (
        'Leave-one-institution-out',
        'Range',
        interval('leave_one_institution_out', 'accepted_precision'),
        interval('leave_one_institution_out', 'eligible_retention'),
        '',
    ),
    (
        'Grade B sensitivity',
        'Logical bounds',
        interval('grade_b_sensitivity', 'accepted_precision'),
        interval('grade_b_sensitivity', 'eligible_retention'),
        '',
    ),
    (
        'Institution-cluster resampling',
        'Percentile range',
        interval('cluster_resampling', 'accepted_precision'),
        interval('cluster_resampling', 'eligible_retention'),
        f"{int(float(cluster_row['replicates'])):,}",
    ),
]

for _, row in robustness[robustness['analysis'] == 'scale_quintile'].iterrows():
    robust_rows.append(
        (
            'Publication-scale quintile',
            f"Quintile {row['group']}",
            pct(row['accepted_precision']),
            pct(row['eligible_retention']),
            '',
        )
    )

robustness_view = pd.DataFrame(
    robust_rows,
    columns=['Diagnostic', 'Group', 'Accepted precision', 'Eligible retention', 'Replicates'],
)
display(robustness_view.set_index(['Diagnostic', 'Group']))


## Reproduced files

The notebook regenerated:

- `results/summary.csv`
- `results/robustness.csv`

The cell below packages only these two CSV files as `reproduced-results.zip` for runtime download.


In [ ]:
import zipfile
from IPython.display import FileLink, display

result_files = [Path('results/summary.csv'), Path('results/robustness.csv')]
zip_path = Path('reproduced-results.zip')
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for result_file in result_files:
        archive.write(result_file, arcname=result_file.name)

try:
    from google.colab import files
    import ipywidgets as widgets

    download_button = widgets.Button(description='Download reproduced-results.zip')
    download_button.on_click(lambda _: files.download(str(zip_path)))
    display(download_button)
except ImportError:
    display(FileLink(zip_path.name))

print('The regenerated files are available above for inspection or download.')
